# Downloading Packages

In [ ]:
import os
import sys
import random
import pickle
import warnings
from datetime import datetime
from pathlib import Path
from itertools import groupby


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader

# --- Local Module Configuration ---
FUNCTIONS_PATH = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
if str(FUNCTIONS_PATH) not in sys.path:
    sys.path.append(str(FUNCTIONS_PATH))

# Importing from your custom files
import ML_functions
from ML_functions import TensorCatchmentDataset, NSELoss
from ML_Models import LSTM_With_Embedding, LSTMModel, HydraModel
from ML_Training import train_model, run_model_pipeline, process_validation_batches, get_basin_names

# Download catchment data

In [ ]:
val_non_res_tensor_data = torch.load('/perm/mokr/CamelsGB_V2/val_non_res_tensor_data.pt')
val_non_res_dataset = TensorCatchmentDataset(val_non_res_tensor_data)

BATCH_SIZE = 1
NUM_WORKERS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val_non_reservoir_dataloader  = DataLoader(val_non_res_dataset, batch_size= BATCH_SIZE, shuffle=False, 
                              num_workers= NUM_WORKERS) # , pin_memory=True, persistent_workers=True

# val_res_tensor_data = torch.load('/perm/mokr/CamelsGB_V2/val_res_tensor_data.pt')
# val_res_dataset = TensorCatchmentDataset(val_res_tensor_data)

In [ ]:
len(val_non_res_dataset)

In [ ]:
# IDs
val_res_tensor_metadata = torch.load('/perm/mokr/CamelsGB_V2/val_res_tensor_metadata.pt')
val_non_res_tensor_metadata = torch.load('/perm/mokr/CamelsGB_V2/val_non_res_tensor_metadata.pt')

def get_consecutive_gauge_indices(metadata):
    """
    Returns a dictionary mapping gauge_id to a slice (start, end)
    Assumes gauge_ids are grouped consecutively.
    """
    gauge_map = {}
    current_idx = 0
    
    # groupby returns the key and an iterator of the group
    for gauge_id, group in groupby(metadata['gauge_id']):
        group_len = len(list(group))
        # Store as range or slice for efficiency
        gauge_map[str(gauge_id)] = range(current_idx, current_idx + group_len)
        current_idx += group_len
        
    return gauge_map

# 1. Pre-calculate the mapping once
gauge_index_map = get_consecutive_gauge_indices(val_res_tensor_metadata)

In [ ]:

def get_consecutive_gauge_indices(metadata):
    """
    Returns a dictionary mapping gauge_id to a slice (start, end)
    Assumes gauge_ids are grouped consecutively.
    """
    gauge_map = {}
    current_idx = 0
    
    # groupby returns the key and an iterator of the group
    for gauge_id, group in groupby(metadata['gauge_id']):
        group_len = len(list(group))
        # Store as range or slice for efficiency
        gauge_map[str(gauge_id)] = range(current_idx, current_idx + group_len)
        current_idx += group_len
        
    return gauge_map

# 1. Pre-calculate the mapping once
gauge_index_map = get_consecutive_gauge_indices(val_res_tensor_metadata)

gauge_loaders = {}
for gid, indices in gauge_index_map.items():
    # Use Subset to point to the specific data for this gauge
    gauge_subset = Subset(val_non_res_dataset, indices)
    
    # Create the loader (Batch size = total gauge samples for single-pass evaluation)
    gauge_loaders[gid] = DataLoader(
        gauge_subset, 
        batch_size=len(indices), 
        shuffle=False
    )



In [ ]:
# Verify that they are the same length

print(len(val_non_res_tensor_data['static']))
print(len(val_res_tensor_metadata['gauge_id']))

## Loading data and finding suitable pairs 

In [ ]:
Camels_Regions = set()
for basin_list in Basin_Lists:
    for rid in basin_list:
        if '_' in rid:
            region = rid.split('_')[0]
            Camels_Regions.add(region)

Camels_Regions = {region.lower() for region in Camels_Regions}
Camels_Regions

In [ ]:
Caravan_Attributes = {}
Hydroatlas_Attributes = {}
Other_Attributes = {}

for Camels_Region in Camels_Regions:
    # download all the Camels_Regions static data
    Caravan_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_caravan_{Camels_Region}.csv')    
    Hydroatlas_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_hydroatlas_{Camels_Region}.csv')
    Other_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_other_{Camels_Region}.csv')
    print(f' {Camels_Region} Done')


print('Static Data Loaded')



basin_caravan_attributes = Caravan_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])
basin_hydroatlas_attributes = Hydroatlas_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])
basin_other_attributes = Other_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])


static_df = pd.concat([basin_caravan_attributes, basin_hydroatlas_attributes, basin_other_attributes], axis=1)
static_df = static_df.select_dtypes(include=['number'])
ordered_static_columns = sorted(static_df.columns.tolist())



In [ ]:
combined_df

In [ ]:
all_data_frames = []
for region, df in Other_Attributes.items():
    all_data_frames.append(df)

combined_df = pd.concat(all_data_frames, ignore_index=True)


# 2. Make a function that takes a value, finds the row with the nearest 'area' value,
#    and returns the 'gauge name' and 'country' elements
def find_nearest_area_info(value, dataframe):
    """
    Finds the row with the nearest 'area' value to the given value and returns
    the 'gauge name' and 'country' elements.

    Args:
        value (float or int): The value to find the nearest 'area' to.
        dataframe (pd.DataFrame): The dataframe to search within. It must
                                  contain 'area', 'gauge name', and 'country' columns.

    Returns:
        tuple: A tuple containing the 'gauge name' and 'country' of the nearest row,
               or (None, None) if the dataframe is empty.
    """
    if dataframe.empty:
        return None, None

    # Calculate the absolute difference between the 'area' column and the given value
    # We create a temporary column 'difference' for this calculation
    dataframe['difference'] = abs(dataframe['area'] - value)

    # Find the index of the row with the minimum difference
    idx_nearest = dataframe['difference'].idxmin()

    # Get the nearest row
    nearest_row = dataframe.loc[idx_nearest]

    # Clean up the temporary 'difference' column (optional, but good practice)
    dataframe.drop(columns=['difference'], inplace=True)

    return nearest_row['gauge_name'], nearest_row['country']

In [ ]:
area_series_list = [attributes['area'] for Camels_Region, attributes in Other_Attributes.items()]
all_areas = pd.concat(area_series_list, ignore_index=True)
sorted_area_counts = all_areas.value_counts().sort_index()

sorted_area_counts[sorted_area_counts > 1]

In [ ]:
# Can extract stuff from /ec/res4/scratch/ecm7072/caravan/attributes
import pandas as pd
import xarray as xr
import os

# --- Load static data ---
Caravan_Attributes = {}
Hydroatlas_Attributes = {}
Other_Attributes = {}

for Camels_Region in Camels_Regions:
    # download all the Camels_Regions static data
    Caravan_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_caravan_{Camels_Region}.csv')    
    Hydroatlas_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_hydroatlas_{Camels_Region}.csv')
    Other_Attributes[Camels_Region] = pd.read_csv(f'/ec/res4/scratch/ecm7072/caravan/attributes/{Camels_Region}/attributes_other_{Camels_Region}.csv')
    print(f' {Camels_Region} Done')


print('Static Done')
    
basin_caravan_attributes = Caravan_Attributes['hysets'].select_dtypes(include=['number'])
basin_hydroatlas_attributes = Hydroatlas_Attributes['hysets'].select_dtypes(include=['number'])
basin_other_attributes = Other_Attributes['hysets'].select_dtypes(include=['number'])


static_df = pd.concat([basin_caravan_attributes, basin_hydroatlas_attributes, basin_other_attributes], axis=1)
static_df = static_df.select_dtypes(include=['number'])
ordered_static_columns = sorted(static_df.columns.tolist())


In [ ]:
static_df[static_df['area'] < 100]

In [ ]:
# Sample data for demonstration (replace with your actual static_df)


static_df = static_df[static_df['area'] > 0]
# End of sample data

min_area = static_df['area'].min()
max_area = static_df['area'].max()

if min_area <= 0:
    min_area = 1e-3 # Small positive value if min_area is non-positive

bins = np.logspace(np.log10(min_area), np.log10(max_area), 50)

plt.figure(figsize=(10, 6))
plt.hist(static_df['area'], bins=bins, edgecolor='black', alpha=0.7)
plt.xscale('log')
plt.xlabel('Area (Log Scale)')
plt.ylabel('Frequency')
plt.title('Histogram of Area with Logarithmic X-axis')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
import os
import xarray as xr

Timeseries = {}

Basin_Lists = [Reservoirs_Train, Reservoirs_Val, Reservoirs_Test,
               NonReservoirs_Train, NonReservoirs_Val, NonReservoirs_Test]

Group_Names = ["Reservoirs_Train", "Reservoirs_Val", "Reservoirs_Test",
               "NonReservoirs_Train", "NonReservoirs_Val", "NonReservoirs_Test"]

Basin_Lists = [Reservoirs_Train, NonReservoirs_Train]

Group_Names = ["Reservoirs_Train", "NonReservoirs_Train"]


total_checked = 0
successful_downloads = 0

for group_name, basin_list in zip(Group_Names, Basin_Lists):
    group_datasets = {}  # Now a dict with basin_id as key

    for basin in basin_list:
        region = basin.split('_')[0]
        nc_path = f'/ec/res4/scratch/ecm7072/caravan/timeseries/netcdf/{region}/{basin}.nc'
        total_checked += 1

        if os.path.exists(nc_path):
            try:
                ds = xr.open_dataset(nc_path)
                group_datasets[basin] = ds
                successful_downloads += 1
            except Exception as e:
                print(f"Error opening {nc_path}: {e}")
        else:
            print(f"Warning: File not found {nc_path}")

        if total_checked % 500 == 0:
            percent = 100 * successful_downloads / total_checked
            print(f"Processed: {total_checked} files — Success: {successful_downloads} ({percent:.1f}%)")

    if group_datasets:
        Timeseries[group_name] = group_datasets
        print(f"{group_name} processed with {len(group_datasets)} datasets")
    else:
        print(f"No data loaded for group {group_name}")




In [ ]:
# Group_Names = ["Reservoirs_Train", "Reservoirs_Val", "Reservoirs_Test",
#                "NonReservoirs_Train", "NonReservoirs_Val", "NonReservoirs_Test"]

# Basin_Lists = [Reservoirs_Train, Reservoirs_Val, Reservoirs_Test,
#                NonReservoirs_Train, NonReservoirs_Val, NonReservoirs_Test]

Split_Metadata = {}  # Final dictionary of DataFrames

for split_name, split in zip(Group_Names, Basin_Lists):
    rows = []
    print(split_name)
    for basin in split:
        region = basin.split('_')[0]

        # Timeseries
        try:
            basin_timeseries = Timeseries[split_name][basin]
            df = basin_timeseries['streamflow'].to_dataframe().dropna()

            # Find breaks in the index
            df['gap'] = (df.index.to_series().diff() != df.index.to_series().diff().min()).cumsum()

            longest_group = df.groupby('gap', group_keys=False).apply(
                lambda x: (x.index[0], x.index[-1], len(x)), 
                include_groups=False
            ).loc[df.groupby('gap').size().idxmax()]
                
            earliest_start = longest_group[0]
            latest_start = longest_group[1] - pd.Timedelta(days=90)

            rows.append({
                'basin': basin,
                'earliest_date': earliest_start,
                'latest_start_for_90day': latest_start
            })

        except Exception as e:
            print(f"Skipping {basin} due to error: {e}")

    # Save DataFrame to dictionary
    Split_Metadata[split_name] = pd.DataFrame(rows).set_index('basin')


# Add total days column to each DataFrame
for split_name in Split_Metadata:
    df = Split_Metadata[split_name]
    df['total_days'] = (df['latest_start_for_90day'] - df['earliest_date']).dt.days + 1



In [ ]:
def sample_basin_date_pairs(split_name, n_samples=1, num_basins = -1):
    """
    Sample uniformly from all possible basin-date pairs in a split
    
    Parameters:
    split_name: str - name of the split to sample from
    n_samples: int - number of samples to draw
    
    Returns:
    list of tuples: [(basin, sampled_date), ...]
    """
    df = Split_Metadata[split_name][0:num_basins]
    
    # Calculate total possible basin-date pairs
    total_pairs = df['total_days'].sum()
    
    # Create cumulative sum for weighted sampling
    df['cumsum_days'] = df['total_days'].cumsum()
    
    samples = []
    for _ in range(n_samples):
        # Random number between 0 and total_pairs-1
        random_idx = np.random.randint(0, total_pairs)
        
        # Find which basin this index falls into
        basin_idx = np.searchsorted(df['cumsum_days'], random_idx, side='right')
        basin = df.index[basin_idx]
        
        # Calculate the offset within that basin's date range
        if basin_idx == 0:
            offset = random_idx
        else:
            offset = random_idx - df['cumsum_days'].iloc[basin_idx - 1]
        
        # Calculate the actual date
        sampled_date = df.loc[basin, 'earliest_date'] + pd.Timedelta(days=offset)
        samples.append((basin, sampled_date))
    
    return samples

In [ ]:
basin_dates_path = '/home/mokr/Reservoir_Modelling/Catchment_Lists/basins_and_dates.pkl'

with open(basin_dates_path, 'rb') as f:
    basin_dates_data = pickle.load(f)
    Split_Metadata = basin_dates_data 

Split_Metadata['NonReservoirs_Train'][0: 10]

In [ ]:
plt.hist(Split_Metadata['Reservoirs_Train']['total_days'], alpha=0.7, label='Reservoirs_Train', bins=30, density = True)
plt.hist(Split_Metadata['NonReservoirs_Train']['total_days'], alpha=0.7, label='NonReservoirs_Train', bins=30, density = True)
plt.xlabel('Total Days')
plt.ylabel('Frequency')
plt.title('Distribution of Total Days')
plt.legend()
plt.show()

In [ ]:
NonReservoirs_Train = Split_Metadata['NonReservoirs_Train'].index.tolist()

In [ ]:
'''with open('/home/mokr/Reservoir_Modelling/Catchment_Lists/basins_and_dates.pkl', 'wb') as f:
    pickle.dump(Split_Metadata, f)'''

## Generating data pair from suitable pairs

In [ ]:
functions_path = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
sys.path.append(str(functions_path))


In [ ]:
import sys
from pathlib import Path

functions_path = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
sys.path.append(str(functions_path))


import ML_functions
from ML_functions import LSTMModel, LSTM_With_Embedding

import importlib
import ML_functions
from ML_functions import LSTMModel, LSTM_With_Embedding

# Reload the module
importlib.reload(ML_functions)

# Re-import the classes after reload
from ML_functions import LSTMModel, LSTM_With_Embedding
import torch
import random

In [ ]:
'''import pickle

# Save to pickle file
with open('/home/mokr/Reservoir_Modelling/Catchment_Lists/basins_and_dates.pkl', 'wb') as f:
    pickle.dump(Split_Metadata, f)'''

In [ ]:


basin_caravan_attributes = Caravan_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])
basin_hydroatlas_attributes = Hydroatlas_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])
basin_other_attributes = Other_Attributes['hysets'].iloc[[0]].select_dtypes(include=['number'])


static_df = pd.concat([basin_caravan_attributes, basin_hydroatlas_attributes, basin_other_attributes], axis=1)
static_df = static_df.select_dtypes(include=['number'])
ordered_static_columns = sorted(static_df.columns.tolist())

start = '1979-01-16 00:00:00'
end = '1979-04-15 00:00:00'

sample_basin_for_temporal_vars = 'hysets_06324500'
ordered_temporal_vars = []
sample_region = sample_basin_for_temporal_vars.split('_')[0]
sample_nc_path = f'/ec/res4/scratch/ecm7072/caravan/timeseries/netcdf/{sample_region}/{sample_basin_for_temporal_vars}.nc'

ds_sample = xr.open_dataset(sample_nc_path)
if 'potential_evaporation_sum_ERA5_LAND' in ds_sample:
    ds_sample = ds_sample.rename({'potential_evaporation_sum_ERA5_LAND': 'potential_evaporation_sum'})
# Exclude streamflow as it's the target
temp_vars = [var for var in ds_sample.data_vars]
ordered_temporal_vars = sorted(temp_vars)


In [ ]:
from sklearn.preprocessing import StandardScaler
import xarray as xr
import numpy as np

def normalize_xarray_dict(xarray_dict, ordered_temporal_vars=None):
    # First pass: preprocess all datasets (rename and select vars if needed)
    preprocessed_dict = {}
    for key, ds in xarray_dict.items():
        ds = ds.copy()
        if 'potential_evaporation_sum_ERA5_LAND' in ds:
            ds = ds.rename({
                'potential_evaporation_sum_ERA5_LAND': 'potential_evaporation_sum'
            })
        if ordered_temporal_vars is not None:
            ds = ds[ordered_temporal_vars]
        preprocessed_dict[key] = ds
    
    # Get all variable names (assuming same variables in each xarray)
    var_names = list(next(iter(preprocessed_dict.values())).data_vars)
    
    # Create scalers for each variable
    scalers = {var: StandardScaler() for var in var_names}
    
    # Fit scalers on concatenated data from all xarrays
    for var in var_names:
        all_data = np.concatenate([
            ds[var].values.flatten()[:, np.newaxis] for ds in preprocessed_dict.values()
        ], axis=0)
        scalers[var].fit(all_data)
        

    # Apply scalers to all datasets
    normalized_dict = {}
    for key, ds in preprocessed_dict.items():
        norm_ds = ds.copy()
        for var in var_names:
            data = ds[var].values
            original_shape = data.shape
            flat_data = data.reshape(-1, 1)
            scaled = scalers[var].transform(flat_data)
            norm_ds[var].values = scaled.reshape(original_shape)
        normalized_dict[key] = norm_ds
        print(f"Fitted scaler for {key}")
    return normalized_dict, scalers
    
normalized_dict, scalers = normalize_xarray_dict(Timeseries['NonReservoirs_Train'])

# joblib.dump(scalers, '/home/mokr/Reservoir_Modelling/nonreservoir_scalers_dict.pkl')

In [ ]:
file_path = '/perm/mokr/Reservoir_Modelling/Normalised_Data/2000_normalized_nonreservoirs_train_data_with_reservoir_scaler.pkl'
with open(file_path, 'rb') as f:
    normalized_nonreservoirs_train_dict = pickle.load(f)

In [ ]:
basin_dates_path = '/home/mokr/Reservoir_Modelling/Catchment_Lists/basins_and_dates.pkl'


with open(basin_dates_path, 'rb') as f:
    basin_dates_data = pickle.load(f)


In [ ]:
# joblib.dump(scaler, '/home/mokr/Reservoir_Modelling/reservoir_static_scalers_dict.pkl')

In [ ]:
import joblib

# Save
# joblib.dump(scalers, '/home/mokr/Reservoir_Modelling/nonreservoir_scalers_dict.pkl')
# joblib.dump(scalers, '/home/mokr/Reservoir_Modelling/reservoir_static_scalers_dict.pkl')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

class NSELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    
    def forward(self, preds: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # preds and target shape: (batch_size, seq_len)
        numerator = torch.sum((target - preds) ** 2, dim=0)
        denominator = torch.sum((target - torch.mean(target, dim=0, keepdim=True)) ** 2, dim=0)
        nse = 1 - numerator / (denominator + self.eps)
        return 1 - nse.mean()  # minimize 1 - NSE

class KGELoss(nn.Module):
    """
    Kling-Gupta Efficiency loss for batched predictions with multiple timesteps.
    KGE = 1 - sqrt((r - 1)^2 + (alpha - 1)^2 + (beta - 1)^2)
    where:
    r = correlation coefficient
    alpha = std(sim)/std(obs)
    beta = mean(sim)/mean(obs)
    
    Handles inputs of shape [batch_size, timesteps]
    """
    def __init__(self, reduction='mean'):
        super(KGELoss, self).__init__()
        self.reduction = reduction
        
    def forward(self, simulated, observed):
        """
        Args:
            simulated (torch.Tensor): Simulated values of shape [batch_size, timesteps]
            observed (torch.Tensor): Observed values of shape [batch_size, timesteps]
        Returns:
            torch.Tensor: KGE loss
        """
        # Ensure the tensors are floating point
        sim = simulated.float()
        obs = observed.float()
        
        # Handle potential NaN values
        eps = 1e-12
        
        # Calculate means along timestep dimension
        sim_mean = torch.mean(sim, dim= 1)  # Shape: [batch_size]
        obs_mean = torch.mean(obs, dim=1)  # Shape: [batch_size]
        
        # Calculate standard deviations along timestep dimension
        sim_std = torch.std(sim, dim=1)  # Shape: [batch_size]
        obs_std = torch.std(obs, dim=1)  # Shape: [batch_size]
        
        # Calculate correlation coefficient for each sample in the batch
        sim_norm = sim - sim_mean.unsqueeze(1)  # Shape: [batch_size, timesteps]
        obs_norm = obs - obs_mean.unsqueeze(1)  # Shape: [batch_size, timesteps]
        
        # Calculate numerator and denominator for correlation
        numerator = torch.sum(sim_norm * obs_norm, dim=1)  # Shape: [batch_size]
        sim_norm_sum = torch.sum(sim_norm ** 2, dim=1)  # Shape: [batch_size]
        obs_norm_sum = torch.sum(obs_norm ** 2, dim=1)  # Shape: [batch_size]
        
        # Add epsilon for numerical stability
        denominator = torch.sqrt(sim_norm_sum * obs_norm_sum + eps)
        r = numerator / denominator  # Shape: [batch_size]
        
        # Calculate alpha (ratio of standard deviations)
        alpha = sim_std / (obs_std + eps)  # Shape: [batch_size]
        
        # Calculate beta (ratio of means)
        # Handle the case where observed mean is zero or very small
        beta = sim_mean / (obs_mean + eps)
        # Calculate KGE for each sample in the batch
        kge = 1 - torch.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)  # Shape: [batch_size]
        
        # Convert to loss (1 - KGE, since KGE's optimal value is 1)
        loss = 1 - kge  # Shape: [batch_size]
        
        if self.reduction == 'mean':
            return torch.mean(loss)
        elif self.reduction == 'sum':
            return torch.sum(loss)
        else:  # 'none'
            return loss




In [ ]:
import pickle 
from tqdm import tqdm
import os
import itertools
import joblib
from datetime import datetime
import gc

import numpy as np
import pandas as pd
import xarray as xr

import sys
from pathlib import Path
functions_path = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
sys.path.append(str(functions_path))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pickle
from tqdm import tqdm
import os
import numpy as np
from pathlib import Path
import sys


import ML_functions
from ML_Training import load_data_tensors, scale_data, create_dataloader, free_memory, train_model, create_dataloaders

# Model & Training Hyperparameters
num_epochs = 10
learning_rate = 1e-3
warmup = 60 # Number of initial time steps to ignore in loss calculation
gradient_accumulation = 1e4 # Perform optimizer step after this many samples
BATCH_SIZE = 64 # The loop is structured for a batch size of 1
static_features_count = 209

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

DATA_DIRECTORY = '/perm/mokr/Reservoir_Modelling/dataset_tensors/'
SCALER_DIRECTORY = '/home/mokr/Reservoir_Modelling/'

all_basin_names = ['Reservoirs_Train_400_800']

all_dataloaders = load_data_tensors(DATA_DIRECTORY, 'Reservoirs_Train_400_800', DEVICE)

In [ ]:

plt.figure(figsize=(10, 6))
data = all_dataloaders[1][:, 13].numpy()
min_area = data.min()
max_area = data.max()

if min_area <= 0:
    min_area = 1e-3 # Small positive value if min_area is non-positive

bins = np.logspace(np.log10(min_area), np.log10(max_area), 50)

plt.figure(figsize=(10, 6))
plt.hist(data, bins=bins, edgecolor='black', alpha=0.7)
plt.xscale('log')
plt.xlabel('Area (Log Scale)')
plt.ylabel('Frequency')
plt.title('Histogram of Area with Logarithmic X-axis')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
area_threshold = 100.0

# Get the specific tensor you want to filter
# This assumes the tensor is the second element in the list
data_tensor = all_dataloaders[1]

# Create a boolean mask for rows where the area is above the threshold
print(f"Filtering for samples with area > {area_threshold}...")
mask = data_tensor[:, 13] > area_threshold

# Apply the mask to select the desired rows
subset_tensor = data_tensor[mask]

# # Define the output path and save the subset to a .npy file
# output_path = 'area_subset.npy'
# # Use .cpu() if your tensor is on a GPU
# np.save(output_path, subset_tensor.cpu().numpy())

# print(f"\nOriginal number of samples: {len(data_tensor)}")
# print(f"Number of samples in subset: {len(subset_tensor)}")
# print(f"Subset saved to: {output_path}")


In [ ]:
import torch
import numpy as np
import gc

# Assume 'temporal_data', 'static_data', and 'targets_data' are your large tensors
print("Saving tensors to disk...")
np.save('/perm/mokr/Reservoir_Modelling/dataset_numpy/Reservoirs_Train_400_800_temporal_data.npy', all_dataloaders[0].cpu().numpy())
np.save('/perm/mokr/Reservoir_Modelling/dataset_numpy/Reservoirs_Train_400_800_static_data.npy', all_dataloaders[1].cpu().numpy())
np.save('/perm/mokr/Reservoir_Modelling/dataset_numpy/Reservoirs_Train_400_800_targets_data.npy', all_dataloaders[2].cpu().numpy())

# IMPORTANT: Delete the original tensors to free up RAM
del all_dataloaders
gc.collect()
print("Original tensors deleted from memory.")

In [ ]:
%whos

# Download Models

In [ ]:
Reservoir_Embedded_LSTM = torch.load('/home/mokr/Reservoir_Modelling/CamelsGB_Models/Reservoir/Reservoir_LSTM_Numlayers1_Embhs64_Embos32.pt', weights_only= False, map_location=torch.device('cpu'))
NonReservoir_Embedded_LSTM = torch.load('/home/mokr/Reservoir_Modelling/CamelsGB_Models/Natural/Natural_LSTM_Numlayers1_Embhs64_Embos32.pt', weights_only= False, map_location=torch.device('cpu'))
Full_Embedded_LSTM = torch.load('/home/mokr/Reservoir_Modelling/CamelsGB_Models/Full/Full_LSTM_Numlayers2_Embhs64_Embos8.pt', weights_only= False, map_location=torch.device('cpu'))


# Hydra_Main = torch.load('/perm/mokr/Reservoir_Modelling/Models/0408_hydra_main_lstm_embedding.pt', weights_only= False, map_location=torch.device('cpu'))
# Hydra_Body = Hydra_Main.body
# Hydra_ResHead = torch.load('/perm/mokr/Reservoir_Modelling/Models/0408_hydra_reshead_lstm_embedding.pt', weights_only= False, map_location=torch.device('cpu'))


# Run Models

In [ ]:
from sklearn.preprocessing import StandardScaler
temporal_scalers = joblib.load('/home/mokr/Reservoir_Modelling/scaler_hydromet.pkl')

scaler_river = StandardScaler()

# Extract the mean, variance, and scale for the -2nd column
scaler_river.mean_ = temporal_scalers.mean_[[-2]]
scaler_river.var_ = temporal_scalers.var_[[-2]]
scaler_river.scale_ = temporal_scalers.scale_[[-2]]


In [ ]:
def calculate_nse(predictions, targets):
    """Calculate Nash-Sutcliffe Efficiency"""
    numerator = torch.mean((predictions - targets) ** 2)
    denominator = torch.mean((targets - torch.mean(targets)) ** 2)
    return 1 - ( (numerator + 1e-3) / (denominator + 1e-3)  )

def calculate_nnse(predictions, targets):
    """Calculate Nash-Sutcliffe Efficiency"""
    numerator = torch.mean((predictions - targets) ** 2)
    denominator = torch.mean((targets - torch.mean(targets)) ** 2)
    nse = 1 - ( (numerator + 1e-3) / (denominator + 1e-3)  )
    return 1 / (2 - nse)


def calculate_kge_components(predictions, targets):
    """Calculate KGE components (alpha, beta, and correlation)"""
    # Correlation coefficient
    pred_mean = torch.mean(predictions)
    target_mean = torch.mean(targets)
    
    numerator = torch.mean((predictions - pred_mean) * (targets - target_mean))
    pred_std = torch.sqrt(torch.mean((predictions - pred_mean) ** 2))
    target_std = torch.sqrt(torch.mean((targets - target_mean) ** 2))
    correlation = numerator / ((pred_std * target_std) + 1e-3)
    
    # Gamma (variability ratio)
    gamma = (pred_std/(pred_mean + 1e-6) + 1e-6) / ( (target_std/ (target_mean + 1e-6)) + 1e-6)
    
    # Beta (bias ratio)
    beta = pred_mean / (target_mean + 1e-3)
    
    return correlation, gamma, beta

def calculate_bias(predictions, targets):
    """Calculate bias (mean difference)"""
    return torch.mean(predictions - targets)

In [ ]:
def validate_model_scores(model_names, models, dataloader, warmup, streamflow_scaler, device='cpu', time_interval=180, max_batches=None):
    # Ensure models is a list
    if not isinstance(models, list):
        models = [models]
    
    # Set all models to evaluation mode
    for model in models:
        if isinstance(model, list):
            # If the item is a list, loop through its sub-models
            for sub_model in model:
                sub_model.eval()
        else:
            # Otherwise, it's a single model
            model.eval()
    
    # Initialize results dictionary to store lists of metrics for each model
    results = {model_name: {
        'NSE': [],
        'NNSE': [],
        'KGE_alpha': [],
        'KGE_beta': [],
        'KGE_correlation': [],
        'KGE': [],
        'bias': []
    } for model_name in model_names}
    
    # Process all validation batches
    batch_outputs = process_validation_batches(
        dataloader, model_names, models, warmup, streamflow_scaler,
        device=device, time_interval=time_interval, max_batches=max_batches
    )
    
    # Calculate metrics for each model and batch
    for model_name in model_names:
        for output_warmup_excluded, target_warmup_excluded in batch_outputs[model_name]:
            nse = calculate_nse(output_warmup_excluded, target_warmup_excluded)
            nnse = calculate_nnse(output_warmup_excluded, target_warmup_excluded)
            bias = calculate_bias(output_warmup_excluded, target_warmup_excluded)
            correlation, alpha, beta = calculate_kge_components(output_warmup_excluded, target_warmup_excluded)
            kge = 1 - np.sqrt((correlation - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
            
            # Store results for this batch
            results[model_name]['NSE'].append(nse.item())
            results[model_name]['NNSE'].append(nnse.item())
            results[model_name]['KGE_alpha'].append(alpha.item())
            results[model_name]['KGE_beta'].append(beta.item())
            results[model_name]['KGE_correlation'].append(correlation.item())
            results[model_name]['KGE'].append(kge.item())
            results[model_name]['bias'].append(bias.item())
    
    return results

In [ ]:
import warnings
# To ignore all DeprecationWarnings (use with extreme caution!)
warnings.filterwarnings("ignore", category=DeprecationWarning)

max_batches = 1e2
warmup = 10

model_names = ['Reservoir Embedded LSTM', 'NonReservoir Embedded LSTM', 'Full Embedded LSTM']
models = [Reservoir_Embedded_LSTM, NonReservoir_Embedded_LSTM, Full_Embedded_LSTM]
validation_results = validate_model_scores(model_names, models, val_non_reservoir_dataloader, warmup, streamflow_scaler = scaler_river, device='cpu', time_interval=10, max_batches= max_batches)

In [ ]:
def display_results_table(results):
    """
    Display validation results as a formatted table.
    """
    import pandas as pd
    
    # Calculate mean scores for each model
    summary = {}
    for model_name, scores in results.items():
        summary[model_name] = {
            'NSE': np.mean(scores['NSE']),
            'NNSE': np.mean(scores['NNSE']),
            'KGE': np.mean(scores['KGE']),
            'KGE_alpha': np.mean(scores['KGE_alpha']),
            'KGE_beta': np.mean(scores['KGE_beta']),
            'KGE_correlation': np.mean(scores['KGE_correlation']),
            'bias': np.mean(scores['bias'])
        }
    
    df = pd.DataFrame(summary).T
    
    # Determine best model for each metric
    best_models = {}
    best_models['NSE'] = df['NSE'].idxmax()
    best_models['NNSE'] = df['NNSE'].idxmax()
    best_models['KGE'] = df['KGE'].idxmax()
    best_models['KGE_correlation'] = df['KGE_correlation'].idxmax()
    best_models['KGE_alpha'] = df['KGE_alpha'].apply(lambda x: abs(x - 1)).idxmin()
    best_models['KGE_beta'] = df['KGE_beta'].apply(lambda x: abs(x - 1)).idxmin()
    best_models['bias'] = df['bias'].apply(abs).idxmin()
    
    # Style the dataframe
    def highlight_best(col):
        best_model = best_models[col.name]
        return ['font-weight: bold' if idx == best_model else '' for idx in col.index]
    
    styled_df = df.style.apply(highlight_best, axis=0).format("{:.4f}")
    
    return styled_df

# Usage
display_results_table(validation_results)

# Scores

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_score_cdfs(results, trim_percent=None):
    """
    Plots the Cumulative Distribution Functions (CDFs) for each score
    for each model. Optionally removes the top and bottom 'trim_percent' of data.

    Args:
        results (dict): A dictionary containing validation scores.
                        Expected format:
                        {
                            'model_0': {'NSE': [...], 'KGE': [...], ...},
                            'model_1': {'NSE': [...], 'KGE': [...], ...},
                            ...
                        }
        trim_percent (float, optional): If specified, the top and bottom
                                        `trim_percent` of the data will be removed
                                        before plotting the CDF. E.g., 1 for 1%.
                                        Defaults to None (no trimming).
    """
    if not results:
        print("No results to plot. The 'results' dictionary is empty.")
        return

    # Get all score names from the first model's results
    # Ensure there's at least one model in results
    if not any(results.values()):
        print("No actual score data in results to plot.")
        return

    first_model_key = next(iter(results))
    if not results[first_model_key]:
        print("First model in results has no scores. Cannot determine score types.")
        return

    score_types = list(results[first_model_key].keys())

    for score_type in score_types:
        plt.figure(figsize=(10, 6))
        
        for model_key, scores in results.items():
            if score_type in scores and scores[score_type]:
                data = np.array(scores[score_type])
                

                if trim_percent is not None and 0 < trim_percent < 50:
                    lower_bound = np.percentile(data, trim_percent)
                    upper_bound = np.percentile(data, 100 - trim_percent) # std_area
                    
                    # Filter data based on the bounds
                    trimmed_data = data[(data >= lower_bound) & (data <= upper_bound)]
                    
                    data = trimmed_data
                    
                elif trim_percent is not None and (trim_percent < 0 or trim_percent >= 50):
                    print(f"Warning: Invalid trim_percent ({trim_percent}) for {score_type} in {model_key}. Must be between 0 and 49.99. Skipping trimming.")

                    # Sort the (potentially trimmed) data for CDF calculation
                sorted_data = np.sort(data)
                
                # Calculate the cumulative probabilities
                cumulative_prob = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
                
                label = f'{model_key} {score_type}'
                if trim_percent is not None:
                    label += f' (trimmed {trim_percent}%)'
                
                plt.plot(sorted_data, cumulative_prob, label=label)
            else:
                print(f"Warning: No data for {score_type} in {model_key}. Skipping plot for this combination.")
        
        title_suffix = f' (trimmed {trim_percent}%)' if trim_percent is not None else ''
        plt.title(f'Cumulative Distribution Function of {score_type}{title_suffix}')
        plt.xlabel(f'{score_type} Value')
        plt.ylabel('Cumulative Probability')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()



In [ ]:
plot_score_cdfs(validation_results, 0.001)

# Hydrographs

In [ ]:
def get_basin_name(basin_idx, csv_base_path="/perm/mokr/Scaled_CamelsGB_V2"):
    # Split by underscore and take the first part as location
    csv_filename = f"normalised_static_data.csv"
    csv_path = os.path.join(csv_base_path, csv_filename)
    df = pd.read_csv(csv_path, engine='python')
    gauge_name = df[df['gauge_id'] == basin_idx]['gauge_name'].values[0]

    return gauge_name

In [ ]:
gauge = '8005'
dataloader = gauge_loaders[gauge]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_outputs = process_validation_batches(
    dataloader, model_names, models, warmup, scaler_river,
    device=DEVICE, time_interval=1, max_batches=max_batches
)

gauge_name = get_basin_name(int(gauge))

In [ ]:
batch_outputs['Reservoir Embedded LSTM'][0][0].shape

In [ ]:
# Create subplots
max_batches = 1
start = 0
end = start + 365

# ECMWF-style colors (vibrant and distinct)
ecmwf_colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFFF33', '#A65628', '#F781BF']
ecmwf_colors = ['#2E86AB', '#F18F01', '#06A77D', '#A23B72']
# Create a single plot
plt.figure(figsize=(15, 8))

# Get target - take the last timestep for each sequence (batch 0)
target = batch_outputs[model_names[0]][0][1][start:end, -1].cpu().numpy()

# Plot target in black
plt.plot(target, label='Target', linewidth=2, color='black')

# Plot each model's output - take the last timestep
for idx, model_name in enumerate(model_names):
    output = batch_outputs[model_name][0][0][start:end, -1].cpu().numpy()
    
    # Customize line style based on model name
    if 'Full' in model_name or 'Hydra' in model_name:
        linestyle = '--'
    else:
        linestyle = '-'
    
    # Use ECMWF color
    color = ecmwf_colors[idx % len(ecmwf_colors)]
    
    plt.plot(output, label=f'{model_name} Output', linewidth=2, linestyle=linestyle, color=color)

# Add labels and title
plt.title(f'Predictions for{gauge_name}')
plt.xlabel('Catchment Index')
plt.ylabel('Streamflow Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gauge = '8005'

max_batches = 10
# This will suppress all warnings that occur while the function is running
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    set_seeds(seed=42)

    reservoir_outputs, targets, gauges, countries = validate_model(Reservoir_Embedded_LSTM, gauge_loaders[gauge], criterion, max_batches, 1, 10, time_interval = 15, share_location = True, Area_df = combined_df.copy())
    unnormalized_reservoir_outputs = streamflow_scaler.inverse_transform(reservoir_outputs.detach().numpy())
    unnormalized_targets = streamflow_scaler.inverse_transform(targets.detach().numpy())

    set_seeds(seed=42)
    
    nonreservoir_outputs, targets = validate_model(NonReservoir_Embedded_LSTM, gauge_loaders[gauge], criterion, max_batches, 1, 10, time_interval = 15, Area_df = combined_df.copy())
    unnormalized_nonreservoir_outputs = streamflow_scaler.inverse_transform(nonreservoir_outputs.detach().numpy())
    unnormalized_targets = streamflow_scaler.inverse_transform(targets.detach().numpy())

    set_seeds(seed=42)
    full_outputs, targets = validate_model(Full_Embedded_LSTM, gauge_loaders[gauge], criterion, max_batches, 1, 10, time_interval = 15, Area_df = combined_df.copy())
    unnormalized_full_outputs = streamflow_scaler.inverse_transform(full_outputs.detach().numpy())

    # set_seeds(seed=42)
    # hydra_res_outputs, targets = validate_model([Hydra_Body, Hydra_ResHead], val_dataloader, criterion, max_batches, 1, 10, time_interval = 15, Area_df = combined_df.copy())
    # unnormalized_hydra_res_outputs = streamflow_scaler.inverse_transform(hydra_res_outputs.detach().numpy())

#


    # Find the basin and list it
    


import matplotlib.pyplot as plt


# Create subplots for i from 1 to 15
fig, axes = plt.subplots(round(max_batches/3), 3, figsize=(45, 20))
axes = axes.flatten()

for i in range(1, max_batches):
    ax = axes[i-1]
    
    # Plot targets and outputs
    ax.plot(unnormalized_targets[i], label='Target', linewidth=2)
    ax.plot(unnormalized_reservoir_outputs[i], label='Reservoir-LSTM Output', linewidth=2)
    ax.plot(unnormalized_nonreservoir_outputs[i], label='NonReservoir-LSTM Output', linewidth=2)
    ax.plot(unnormalized_full_outputs[i], label='Full-LSTM Output', linewidth=2, linestyle='--')
    ax.plot(unnormalized_hydra_res_outputs[i], label='Hydra Output', linewidth=2, linestyle='--')

    # Add labels and title
    ax.set_title(f'Plot for gauge {gauges[i]}')
    ax.set_xlabel('Time Step') # Changed from Index to Time Step for better clarity
    ax.set_ylabel('Streamflow Value') # More descriptive label
    ax.legend()
    ax.grid(True, alpha=0.3)


plt.tight_layout()
plt.show()


In [ ]:

def calculate_kge(predictions, targets):

    eps = 1e-6
    """Calculate KGE components (alpha, beta, and correlation)"""
    # Correlation coefficient
    pred_mean = np.mean(predictions)
    target_mean = np.mean(targets)
    
    numerator = np.mean((predictions - pred_mean) * (targets - target_mean))
    pred_std = np.sqrt(np.mean((predictions - pred_mean) ** 2))
    target_std = np.sqrt(np.mean((targets - target_mean) ** 2))
    correlation = numerator / ((pred_std * target_std) + eps)
    
    # Alpha (variability ratio)
    gamma = ( pred_std/(pred_mean + eps)) / (target_std / (target_mean + eps) )
    
    # Beta (bias ratio)
    beta = pred_mean / (target_mean + eps)

    kge = 1 - np.sqrt((correlation - 1)**2 + (gamma - 1)**2 + (beta - 1)**2)
    # print(gamma, beta, correlation)
    return kge

def calculate_nse(predictions, targets):
    eps = 1e-8
    """Calculate Nash-Sutcliffe Efficiency"""
    numerator = np.mean((predictions - targets) ** 2)
    denominator = np.mean((targets - np.mean(targets)) ** 2)
    return 1 - ( (numerator + eps)/ (denominator + eps)  )

In [ ]:
Reservoir_Embedded_LSTM = torch.load('/perm/mokr/Reservoir_Modelling/Models/Res_3107_embedded_lstm.pt', weights_only= False, map_location=torch.device('cpu'))
NonReservoir_Embedded_LSTM = torch.load('/perm/mokr/Reservoir_Modelling/Models/NonReservoir_embedded_lstm_3107.pt', weights_only= False, map_location=torch.device('cpu'))
Full_Embedded_LSTM = torch.load('/perm/mokr/Reservoir_Modelling/Models/full_embedded_lstm_3107.pt', weights_only= False, map_location=torch.device('cpu'))

Hydra_Main = torch.load('/perm/mokr/Reservoir_Modelling/Models/0408_hydra_main_lstm_embedding.pt', weights_only= False, map_location=torch.device('cpu'))
Hydra_Body = Hydra_Main.body
Hydra_ResHead = torch.load('/perm/mokr/Reservoir_Modelling/Models/0408_hydra_reshead_lstm_embedding.pt', weights_only= False, map_location=torch.device('cpu'))


In [ ]:
import warnings

max_batches = 15
seed = 44
warmup = 0
loss_interval = 2
# This will suppress all warnings that occur while the function is running
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    set_seeds(seed= seed)

    
    # Reservoir model
    reservoir_outputs, targets, gauges, countries = validate_model(Reservoir_Embedded_LSTM, val_dataloader, criterion, max_batches, warmup, 10, time_interval = 15, share_location = True)
    unnormalized_reservoir_outputs = streamflow_scaler.inverse_transform(reservoir_outputs.detach().numpy())
    unnormalized_reservoir_outputs = np.clip(unnormalized_reservoir_outputs, a_min=0, a_max=None)  # <-- Clip below 0
    unnormalized_targets = streamflow_scaler.inverse_transform(targets.detach().numpy())
    
    # Non-reservoir model
    set_seeds(seed=seed)
    nonreservoir_outputs, targets = validate_model(NonReservoir_Embedded_LSTM, val_dataloader, criterion, max_batches, warmup, loss_interval, time_interval=15)
    unnormalized_nonreservoir_outputs = streamflow_scaler.inverse_transform(nonreservoir_outputs.detach().numpy())
    unnormalized_nonreservoir_outputs = np.clip(unnormalized_nonreservoir_outputs, a_min=0, a_max=None)  # <-- Clip
    
    # Full model
    set_seeds(seed=seed)
    full_outputs, targets = validate_model(Full_Embedded_LSTM, val_dataloader, criterion, max_batches, warmup, loss_interval, time_interval=15)
    unnormalized_full_outputs = streamflow_scaler.inverse_transform(full_outputs.detach().numpy())
    unnormalized_full_outputs = np.clip(unnormalized_full_outputs, a_min=0, a_max=None)  # <-- Clip
    
    # Hydra model
    set_seeds(seed=seed)
    hydra_res_outputs, targets = validate_model([Hydra_Body, Hydra_ResHead], val_dataloader, criterion, max_batches, warmup, loss_interval, time_interval=15)
    unnormalized_hydra_res_outputs = streamflow_scaler.inverse_transform(hydra_res_outputs.detach().numpy())
    unnormalized_hydra_res_outputs = np.clip(unnormalized_hydra_res_outputs, a_min=0, a_max=None)  # <-- Clip
    
    

    # set_seeds(seed=42)
    # hydra_res_outputs, targets = validate_model(Hydra_Main, val_dataloader, criterion, max_batches, warmup, loss_interval, time_interval = 15)
    # unnormalized_hydra_res_outputs = streamflow_scaler.inverse_transform(hydra_res_outputs.detach().numpy())


import matplotlib.pyplot as plt


for i in range(1, max_batches):

    plt.figure(figsize=(12, 6))

    kge_reservoir = calculate_kge(unnormalized_reservoir_outputs[i], unnormalized_targets[i])
    kge_nonreservoir = calculate_kge(unnormalized_nonreservoir_outputs[i], unnormalized_targets[i])
    kge_full = calculate_kge(unnormalized_full_outputs[i], unnormalized_targets[i])
    kge_hydra = calculate_kge(unnormalized_hydra_res_outputs[i], unnormalized_targets[i])
     
    nse_reservoir = calculate_nse(unnormalized_reservoir_outputs[i], unnormalized_targets[i])
    nse_nonreservoir = calculate_nse(unnormalized_nonreservoir_outputs[i], unnormalized_targets[i])
    nse_full = calculate_nse(unnormalized_full_outputs[i], unnormalized_targets[i])
    nse_hydra = calculate_nse(unnormalized_hydra_res_outputs[i], unnormalized_targets[i])

    model_names = ['Reservoir_Embedded_LSTM', 'NonReservoir_Embedded_LSTM', 'Full_Embedded_LSTM', 'Hydra_ResHead', 'Hydra_Main']
    colors = ['orange', 'green', 'red', 'purple', 'blue']
    
    # Plot targets and outputs with metrics in legend labels
    plt.plot(unnormalized_targets[i], label='Target', color='black', linewidth=2)
    plt.plot(unnormalized_reservoir_outputs[i], 
             label=f'Reservoir-LSTM (NSE: {nse_reservoir:.3f}, KGE: {kge_reservoir:.3f})', 
             color='orange', linestyle='--', linewidth=2)
    plt.plot(unnormalized_nonreservoir_outputs[i], 
             label=f'NonReservoir-LSTM (NSE: {nse_nonreservoir:.3f}, KGE: {kge_nonreservoir:.3f})', 
             color='green', linestyle='--', linewidth=2)
    plt.plot(unnormalized_full_outputs[i], 
             label=f'Full-LSTM (NSE: {nse_full:.3f}, KGE: {kge_full:.3f})', 
             color = 'red', linewidth=2, linestyle='--')
    plt.plot(unnormalized_hydra_res_outputs[i], 
             label=f'Hydra (NSE: {nse_hydra:.3f}, KGE: {kge_hydra:.3f})', 
             color = 'purple', linewidth=2, linestyle='--')
    
    # Add labels and title
    plt.title(f"Streamflow for Basin: {gauges[i]}")
    plt.xlabel("Date")
    plt.ylabel("Specific Discharge ($ms^{-1}$)")
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

    

In [ ]:

# samples = sample_basin_date_pairs('NonReservoirs_Train', n_samples=10, num_basins=90)

for basin in  ['hysets_11443000']: # basins hysets_12457000 1976 hysets_12199000 hysets_11443000 camelsch_3019
    region = basin.split('_')[0] 
    
    # Static data
    
    
    basin_caravan_attributes = Caravan_Attributes[region][Caravan_Attributes[region]['gauge_id'] == basin].copy()
    basin_hydroatlas_attributes = Hydroatlas_Attributes[region][Hydroatlas_Attributes[region]['gauge_id'] == basin].copy()
    basin_other_attributes = Other_Attributes[region][Other_Attributes[region]['gauge_id'] == basin].copy()

    static_df = pd.concat([basin_caravan_attributes, basin_hydroatlas_attributes, basin_other_attributes], axis=1)
    static_df = static_df.select_dtypes(include=['number'])
    static_df = static_df.loc[:, static_df.columns.isin(ordered_static_cols)]
    static_df = static_df.reindex(columns=ordered_static_cols) # Ensure consistent column order
    static_input = torch.tensor(static_df.values, dtype=torch.float32)  # shape: (1, num_features)

    # Temporal data
    start = pd.to_datetime('1956-01-01')
    end = start + pd.Timedelta(days=365)
    basin_temporal = group_datasets[basin].sel(date=slice(start, end))

    
    plot_dates = basin_temporal.date.values 

    if 'potential_evaporation_sum_ERA5_LAND' in basin_temporal:
        basin_temporal = basin_temporal.rename({
            'potential_evaporation_sum_ERA5_LAND': 'potential_evaporation_sum'})
        
    basin_temporal_input = basin_temporal[ordered_temporal_vars]
    targets = basin_temporal['streamflow']
    # temporal_input = basin_temporal_input.drop_vars('streamflow')

    # Prepare tensors for the model
    temporal_np = basin_temporal_input.to_array().transpose('date', 'variable').values
    temporal_input = torch.tensor(temporal_np, dtype=torch.float32).unsqueeze(0)
    target = torch.tensor(targets.values, dtype=torch.float32).unsqueeze(0)

    temporal_scaled, static_scaled, targets_scaled, _ = scale_data(temporal_tensor=temporal_input,
                                                                    static_tensor=static_input,
                                                                    targets_tensor=target,
                                                                    scaler_dir= SCALER_DIRECTORY,
                                                                    device= 'cpu')

    
    models = [Reservoir_Embedded_LSTM, NonReservoir_Embedded_LSTM, Full_Embedded_LSTM, [Hydra_Body, Hydra_ResHead], Hydra_Main]
    model_names = ['Reservoir_Embedded_LSTM', 'NonReservoir_Embedded_LSTM', 'Full_Embedded_LSTM', 'Hydra_ResHead', 'Hydra_Main']
    colors = ['orange', 'green', 'red', 'purple', 'blue']
    
    models = [NonReservoir_Embedded_LSTM, Full_Embedded_LSTM, Hydra_Main]
    model_names = ['NonReservoir_Embedded_LSTM', 'Full_Embedded_LSTM', 'Hydra_Main']
    colors = [ 'green', 'red', 'blue']

        


    
    plt.figure(figsize=(12, 6))
    
    # Plot each model
    for i, (model, name, color) in enumerate(zip(models, model_names, colors)):
        # Run the model pipeline
        if isinstance(model, list):  # Handle the Hydra case
            model_outputs = run_model_pipeline(model, static_scaled, temporal_scaled).squeeze(-1)
        else:
            model_outputs = run_model_pipeline(model, static_scaled, temporal_scaled).squeeze(-1)
        
        # Unnormalize outputs
        unnormalized_outputs = streamflow_scaler.inverse_transform(model_outputs.detach().numpy())
        unnormalized_outputs = np.clip(unnormalized_outputs, a_min=0, a_max=None)
        
        # Plot the line
        kge = calculate_nse(unnormalized_outputs, targets.values)
        calculate_kge(unnormalized_outputs, targets.values)
        # Plot the line with KGE in the label
        plt.plot(plot_dates, unnormalized_outputs.squeeze(), 
                 linestyle = '--', label=f'{name} (NSE: {kge:.3f})', color=color)

    # Plot observed streamflow
    plt.plot(plot_dates, targets.values, label='Observed Streamflow', color='black', linewidth=2)
    
    plt.xlabel("Date")
    plt.ylabel("Specific Discharge ($ms^{-1}$)")
    plt.legend()

    plt.title(f"Predicted vs Target Streamflow for Basin: {basin}, Start Date: {start.strftime('%Y-%m-%d')}")
    
    # A more robust title if you comment out predictions:
    plt.title(f"Observed Streamflow for Basin: {basin}\nStart Date: {start.strftime('%Y-%m-%d')}")

    
    plt.grid(True, which='both', linestyle='--', linewidth=0.5) # Add grid for better readability
    plt.tight_layout()
    plt.show()
